# Aula 2 - Classificação de imagens com CNN e ResNet

> Este notebook é uma demonstração educacional e não constitui um sistema de diagnóstico médico.

Anteriormente vimos como uma rede simples que recebeu features numéricas prontas pôde aprender a classificar dados tabulares em duas classes.

Agora construiremos redes capazes de **aprender as features diretamente dos pixels** para classificar imagens em quatro classes.

Primeiro introduziremos o conceito de convoluções e redes convolucionais (CNNs). Em seguida, implementaremos uma CNN pequena e, depois, entenderemos conexões residuais e compararemos transfer learning com ResNet-18 e ResNet-50.

## Por que não usar apenas uma MLP?

Uma imagem RGB de `224 × 224` pixels é representada por:

$$3 \times 224 \times 224 = 150.528$$

valores numéricos, um para cada combinação de canal, linha e coluna.

![RGB Channels](../../assets/CNN-model-sequentially-identifies-patterns-processes-images.png)

Como visto em [Understanding Convolution Neural Network (CNN) Architecture - Deep Learning](https://www.ksolves.com/blog/artificial-intelligence/understanding-convolution-neural-network-architecture)

Para receber essa imagem, uma MLP precisaria primeiro transformá-la em um único vetor com 150.528 elementos. Esse processo, chamado **flattening, elimina explicitamente a organização espacial da imagem**: o modelo deixa de saber quais pixels estavam próximos, quais formavam uma borda e quais pertenciam à mesma região.

Além disso, conectar esses 150.528 valores a uma camada com apenas 128 neurônios já exigiria mais de **19 milhões de pesos**:

$$150.528 \times 128 = 19.267.584$$

Uma CNN é mais adequada porque processa pequenas regiões da imagem por vez. Seus filtros compartilham os mesmos pesos em diferentes posições, permitindo reconhecer padrões como bordas, contrastes e texturas independentemente de onde eles aparecem.

![CNN Architecture](../../assets/CNN-Model-Architecture.png)

Como visto em [Understanding Convolution Neural Network (CNN) Architecture - Deep Learning](https://www.ksolves.com/blog/artificial-intelligence/understanding-convolution-neural-network-architecture)

Assim, a CNN:

- Preserva melhor as relações espaciais entre os pixels;
- Utiliza muito menos parâmetros;
- Reconhece o mesmo padrão em diferentes regiões;
- Combina padrões simples para aprender estruturas progressivamente mais complexas.

Uma MLP ainda pode classificar imagens, mas geralmente é menos eficiente e ignora uma informação essencial: **a posição e a vizinhança dos pixels também possuem significado**.

## Convolução

Um **kernel** é uma pequena matriz de pesos que percorre a imagem. Em cada posição, ele combina os pixels de uma região local e produz um valor em um novo mapa de características:

$$z_{i,j} = \sum_{c,u,v} K_{c,u,v} X_{c,i+u,j+v} + b$$

Os pesos do kernel são aprendidos por backpropagation, assim como os pesos das camadas lineares da MLP. As primeiras camadas costumam responder a padrões simples, enquanto camadas posteriores combinam esses padrões em representações mais abstratas.

### ReLU, pooling e canais

- **ReLU** adiciona não linearidade.
- **Pooling** reduz as dimensões espaciais, preservando respostas importantes.
- **Canais** representam diferentes mapas de características aprendidos.
- **Batch normalization** estabiliza as ativações e facilita o treinamento.

Nossa CNN aumentará os canais de `3 → 16 → 32 → 64` enquanto reduz altura e largura.

### Leitura adicional

Para uma explicação visual e intuitiva de CNN's, recomenda-se:

- [But what is a convolution? - 3Blue1Brown](https://www.youtube.com/watch?v=KuXjwB4LzSA)

## Bibliotecas, seed e dispositivo

In [ ]:
from copy import deepcopy
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.preprocessing import label_binarize
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights, ResNet50_Weights

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Dispositivo:", device)
print("CUDA:", torch.version.cuda)

## Manifesto preparado

O notebook [`01-prepare-images.pt-br.ipynb`](01-prepare-images.pt-br.ipynb) criou um manifesto com caminhos, rótulos e splits. O conjunto de teste já está definido, mas só será usado na avaliação final.

In [ ]:
def find_project_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Raiz do projeto não encontrada.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
MANIFEST_PATH = DATA_DIR / "alzheimer_mri_manifest.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "lesson-02"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not MANIFEST_PATH.exists():
    raise FileNotFoundError("Execute 01-prepare-images.pt-br.ipynb antes deste notebook.")

manifest = pd.read_csv(MANIFEST_PATH)

CLASS_NAMES = (
    manifest[["label", "class_name"]]
    .drop_duplicates().sort_values("label")["class_name"].tolist()
)

NUM_CLASSES = len(CLASS_NAMES)

pd.crosstab(manifest["split"], manifest["class_name"])

## Transformações

As imagens de treino recebem pequenas variações aleatórias de recorte e rotação. Isso é **data augmentation**: novas versões plausíveis são geradas durante o treino, sem alterar os arquivos originais.

![Data Augmentation](../../assets/data-augmentation-random.png)

Como visto em [O que é aumento de dados? ](https://www.ibm.com/br-pt/think/topics/data-augmentation)

Validação e teste usam transformações determinísticas. Os valores de normalização são os mesmos usados nos pesos ImageNet das ResNets, permitindo que os três modelos recebam entradas equivalentes.

In [ ]:
IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.90, 1.00)),
    transforms.RandomRotation(degrees=5),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

evaluation_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

## Dataset com carregamento lazy

O objeto armazena somente o manifesto. Cada imagem é aberta em `__getitem__`, no momento em que seu batch é solicitado. Isso economiza memória e permite que o augmentation seja sorteado novamente a cada época.

In [ ]:
class AlzheimerImageDataset(Dataset):
    def __init__(self, frame, data_dir, transform):
        self.frame = frame.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        record = self.frame.iloc[index]
        image_path = self.data_dir / record["relative_path"]
        with Image.open(image_path) as image:
            image = image.convert("RGB")
            image = self.transform(image)
        label = torch.tensor(int(record["label"]), dtype=torch.long)
        return image, label


train_frame = manifest.query("split == 'train'").copy()
val_frame = manifest.query("split == 'validation'").copy()
test_frame = manifest.query("split == 'test'").copy()

train_dataset = AlzheimerImageDataset(train_frame, DATA_DIR, train_transform)
val_dataset = AlzheimerImageDataset(val_frame, DATA_DIR, evaluation_transform)
test_dataset = AlzheimerImageDataset(test_frame, DATA_DIR, evaluation_transform)

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 0  # configuração segura para notebooks no Windows e no Linux
generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), generator=generator,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

images, labels = next(iter(train_loader))
print("Batch de imagens:", images.shape)
print("Batch de rótulos:", labels.shape)

## Pesos das classes

Calcularemos pesos inversamente proporcionais à frequência de cada classe no treino. Com o manifesto balanceado produzido pelo notebook anterior, esses pesos serão iguais ou muito próximos; o cálculo permanece útil caso `MAX_IMAGES_PER_CLASS` ou o dataset sejam alterados.

In [ ]:
class_counts = train_frame["label"].value_counts().sort_index()
class_weights = len(train_frame) / (NUM_CLASSES * class_counts.to_numpy())
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

pd.DataFrame({
    "classe": CLASS_NAMES,
    "amostras_treino": class_counts.to_numpy(),
    "peso_na_loss": class_weights.cpu().numpy(),
})

## CNN simples

A primeira arquitetura é pequena o suficiente para ser compreendida por inteiro:

```text
Imagem: 3 × 224 × 224
  ↓ Conv 3→16 + BatchNorm + ReLU + MaxPool
  ↓ Conv 16→32 + BatchNorm + ReLU + MaxPool
  ↓ Conv 32→64 + BatchNorm + ReLU + MaxPool
  ↓ Adaptive Average Pooling
  ↓ Linear 64→4
```

O `AdaptiveAvgPool2d` resume cada canal em um único valor. Isso evita uma camada linear enorme e permite receber imagens de dimensões espaciais variadas.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        return self.classifier(x)


def parameter_counts(model):
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    return total, trainable


simple_cnn = SimpleCNN(NUM_CLASSES)
total, trainable = parameter_counts(simple_cnn)
print(simple_cnn)
print(f"Parâmetros totais: {total:,}")
print(f"Parâmetros treináveis: {trainable:,}")

### Logits e CrossEntropyLoss

Assim como na MLP, a última camada produz quatro logits. A `CrossEntropyLoss` aplica internamente as operações necessárias para comparar esses scores com o rótulo correto. O `Softmax` será usado somente na avaliação, quando quisermos interpretar probabilidades.

## Treinamento reutilizável

O mesmo ciclo será usado para a CNN simples, a ResNet-18 e a ResNet-50: treino, validação, armazenamento do melhor estado e early stopping. A validação orienta o treinamento; o teste ainda não é acessado.

In [ ]:
def keep_frozen_batchnorm_in_eval(model):
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            parameters = list(module.parameters())
            if parameters and not any(parameter.requires_grad for parameter in parameters):
                module.eval()


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    if training:
        model.train()
        keep_frozen_batchnorm_in_eval(model)
    else:
        model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for inputs, targets in loader:
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            logits = model(inputs)
            loss = criterion(logits, targets)
            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * targets.size(0)
        total_correct += (logits.argmax(dim=1) == targets).sum().item()
        total_samples += targets.size(0)

    return total_loss / total_samples, total_correct / total_samples


def train_model(model, optimizer, epochs, name, patience=4):
    model = model.to(device)
    history = {"train_loss": [], "val_loss": [], "train_accuracy": [], "val_accuracy": []}
    best_state = deepcopy(model.state_dict())
    best_val_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):
        train_loss, train_accuracy = run_epoch(model, train_loader, optimizer)
        val_loss, val_accuracy = run_epoch(model, val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_accuracy"].append(val_accuracy)

        print(
            f"{name} | época {epoch:02d}/{epochs} | "
            f"loss {train_loss:.4f}/{val_loss:.4f} | "
            f"acurácia {train_accuracy:.2%}/{val_accuracy:.2%}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print("Early stopping: a loss de validação parou de melhorar.")
                break

    model.load_state_dict(best_state)
    torch.save(best_state, OUTPUT_DIR / f"{name}.pt")
    return model, history, best_val_loss


def plot_history(history, title):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, history["train_loss"], label="Treino")
    axes[0].plot(epochs, history["val_loss"], label="Validação")
    axes[0].set_title("Loss")
    axes[1].plot(epochs, history["train_accuracy"], label="Treino")
    axes[1].plot(epochs, history["val_accuracy"], label="Validação")
    axes[1].set_title("Acurácia")
    for axis in axes:
        axis.set_xlabel("Época")
        axis.grid(alpha=0.3)
        axis.legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
cnn_optimizer = torch.optim.AdamW(simple_cnn.parameters(), lr=1e-3, weight_decay=1e-4)
simple_cnn, cnn_history, cnn_best_val_loss = train_model(
    simple_cnn, cnn_optimizer, epochs=15, name="simple_cnn", patience=4
)
plot_history(cnn_history, "CNN simples")

## Da CNN para uma Rede Neural Residual (ResNet)

Adicionar mais camadas nem sempre melhora uma rede neural. Em arquiteturas muito profundas, o sinal precisa atravessar muitas transformações e o gradiente propagado de volta durante o treinamento pode se tornar cada vez menor. Esse fenômeno, conhecido como **desaparecimento do gradiente** (*vanishing gradient*), dificulta o aprendizado das primeiras camadas.

Uma **Rede Neural Residual**, ou **ResNet**, reduz essa dificuldade usando atalhos chamados **conexões de salto** (*skip connections*). Esses atalhos permitem que a entrada de um bloco avance diretamente por algumas camadas e seja somada ao resultado das transformações realizadas por elas.

![Arquitetura de uma ResNet](../../assets/image-19.png)

Como visto em: [Residual Neural Networks - What You Need to Know](https://datascience.eu/machine-learning/an-overview-of-resnet-and-its-variants/).

Em um bloco residual, a relação pode ser representada por:

$$y = F(x) + x$$

Nessa expressão, `x` é a entrada do bloco e `F(x)` representa as transformações aprendidas pelas camadas internas. Em vez de aprender toda a transformação desejada, o bloco precisa aprender apenas uma **correção residual**. Se nenhuma correção for necessária, aproximar `F(x) = 0` preserva a entrada pelo atalho.

Além de facilitar a passagem da informação, o atalho cria um caminho mais direto para os gradientes durante a retropropagação. Isso torna o treinamento de redes profundas mais estável, embora não elimine sozinho todos os problemas relacionados ao gradiente.

## Transfer learning com ResNet-18

A ResNet-18 foi originalmente treinada no **ImageNet**, um grande conjunto de imagens naturais organizado em 1.000 classes. Durante esse treinamento, a rede ajustou milhões de parâmetros para reconhecer padrões visuais úteis na distinção de objetos. Ao usar `ResNet18_Weights.DEFAULT`, carregamos esses parâmetros já aprendidos, chamados de **pesos pré-treinados**, em vez de iniciar a rede com pesos aleatórios.

As representações aprendidas são hierárquicas. As primeiras camadas geralmente detectam características visuais simples e genéricas, como bordas, orientações, contrastes e texturas. As camadas seguintes combinam esses elementos em padrões progressivamente mais complexos, enquanto as últimas camadas ficam mais especializadas nas classes usadas durante o treinamento original.

O reaproveitamento desses pesos é chamado de **transfer learning**. A ideia é usar o conhecimento visual adquirido em uma tarefa ampla como **ponto de partida** para uma nova tarefa. Isso costuma reduzir o tempo de treinamento e a quantidade de dados necessária quando comparado ao treinamento completo da mesma arquitetura a partir de pesos aleatórios.

Existe, porém, uma diferença de domínio: o ImageNet contém principalmente fotografias coloridas de objetos cotidianos, enquanto nosso dataset contém imagens médicas originalmente em tons de cinza. Portanto, os filtros genéricos das primeiras camadas podem ser úteis, mas as representações mais especializadas talvez precisem ser adaptadas durante o fine-tuning.

### Congelamento do backbone e substituição da cabeça

Podemos dividir a ResNet em duas partes conceituais:

- o **backbone** reúne os blocos convolucionais responsáveis por extrair representações da imagem;
- a **cabeça de classificação** (*classification head*) é a camada final `fc`, que transforma essas representações em logits para as classes da tarefa.

Na primeira fase, definimos `requires_grad = False` para os parâmetros do backbone. Com isso, seus pesos pré-treinados permanecem fixos durante a retropropagação e não são atualizados pelo otimizador. A camada `fc` original, criada para as 1.000 classes do ImageNet, é então substituída por uma nova camada com `NUM_CLASSES` saídas.

Como a nova cabeça é criada depois do congelamento, seus parâmetros continuam treináveis. Nessa etapa, a ResNet funciona como um **extrator de features fixo**: o backbone produz representações e somente a cabeça aprende a relacioná-las às quatro classes do nosso dataset. Mais adiante, descongelaremos o último bloco do backbone para realizar um ajuste fino dessas representações.

In [ ]:
resnet18 = models.resnet18(weights=ResNet18_Weights.DEFAULT)
for parameter in resnet18.parameters():
    parameter.requires_grad = False

resnet18.fc = nn.Linear(resnet18.fc.in_features, NUM_CLASSES)
total, trainable = parameter_counts(resnet18)
print(f"Parâmetros totais: {total:,}")
print(f"Parâmetros treináveis na fase 1: {trainable:,}")
print(f"Proporção treinável: {trainable / total:.3%}")

In [ ]:
resnet18_head_optimizer = torch.optim.AdamW(
    resnet18.fc.parameters(), lr=1e-3, weight_decay=1e-4
)
resnet18, resnet18_head_history, resnet18_head_best_val_loss = train_model(
    resnet18, resnet18_head_optimizer, epochs=8, name="resnet18_head", patience=3
)
resnet18_head_state = deepcopy(resnet18.state_dict())
plot_history(resnet18_head_history, "ResNet-18 — treinamento da cabeça")

### Fine-tuning do último bloco

Agora descongelaremos somente o último bloco residual (`layer4`). Usaremos learning rates menores para adaptar representações de alto nível sem modificar bruscamente os filtros pré-treinados. Se a validação piorar, manteremos o estado obtido na fase anterior.

In [ ]:
RUN_FINE_TUNING = True

if RUN_FINE_TUNING:
    for parameter in resnet18.layer4.parameters():
        parameter.requires_grad = True

    resnet18_fine_tuning_optimizer = torch.optim.AdamW([
        {"params": resnet18.layer4.parameters(), "lr": 1e-4},
        {"params": resnet18.fc.parameters(), "lr": 5e-4},
    ], weight_decay=1e-4)

    resnet18, resnet18_fine_history, resnet18_fine_best_val_loss = train_model(
        resnet18, resnet18_fine_tuning_optimizer, epochs=5,
        name="resnet18_finetuned", patience=3
    )

    if resnet18_fine_best_val_loss > resnet18_head_best_val_loss:
        print("O fine-tuning não melhorou a validação; restaurando a melhor cabeça.")
        resnet18.load_state_dict(resnet18_head_state)
    else:
        plot_history(resnet18_fine_history, "ResNet-18 — fine-tuning do layer4")

## Transfer learning com ResNet-50

A ResNet-50 é mais profunda e usa blocos residuais do tipo **bottleneck**, enquanto a ResNet-18 usa blocos básicos. Ela possui muito mais parâmetros e maior custo computacional, mas sua capacidade adicional não garante melhor generalização neste dataset relativamente pequeno.

Para tornar a comparação consistente, repetiremos o mesmo protocolo: pesos ImageNet, treinamento inicial somente da cabeça e fine-tuning apenas do `layer4`, com as mesmas épocas, learning rates e regras de early stopping. A ResNet-50 também consome mais memória; se necessário, reduza `BATCH_SIZE` antes de criar os DataLoaders.

### Congelamento do backbone e substituição da cabeça

In [ ]:
resnet50 = models.resnet50(weights=ResNet50_Weights.DEFAULT)
for parameter in resnet50.parameters():
    parameter.requires_grad = False

resnet50.fc = nn.Linear(resnet50.fc.in_features, NUM_CLASSES)
total, trainable = parameter_counts(resnet50)
print(f"Parâmetros totais: {total:,}")
print(f"Parâmetros treináveis na fase 1: {trainable:,}")
print(f"Proporção treinável: {trainable / total:.3%}")

In [ ]:
resnet50_head_optimizer = torch.optim.AdamW(
    resnet50.fc.parameters(), lr=1e-3, weight_decay=1e-4
)
resnet50, resnet50_head_history, resnet50_head_best_val_loss = train_model(
    resnet50, resnet50_head_optimizer, epochs=8, name="resnet50_head", patience=3
)
resnet50_head_state = deepcopy(resnet50.state_dict())
plot_history(resnet50_head_history, "ResNet-50 — treinamento da cabeça")

### Fine-tuning do último bloco

In [ ]:
if RUN_FINE_TUNING:
    for parameter in resnet50.layer4.parameters():
        parameter.requires_grad = True

    resnet50_fine_tuning_optimizer = torch.optim.AdamW([
        {"params": resnet50.layer4.parameters(), "lr": 1e-4},
        {"params": resnet50.fc.parameters(), "lr": 5e-4},
    ], weight_decay=1e-4)

    resnet50, resnet50_fine_history, resnet50_fine_best_val_loss = train_model(
        resnet50, resnet50_fine_tuning_optimizer, epochs=5,
        name="resnet50_finetuned", patience=3
    )

    if resnet50_fine_best_val_loss > resnet50_head_best_val_loss:
        print("O fine-tuning não melhorou a validação; restaurando a melhor cabeça.")
        resnet50.load_state_dict(resnet50_head_state)
    else:
        plot_history(resnet50_fine_history, "ResNet-50 — fine-tuning do layer4")

## Avaliação final no teste

Somente agora os três modelos recebem o conjunto de teste. Além da acurácia, usaremos métricas macro, que atribuem a mesma importância a cada classe, e balanced accuracy, que continua comparável mesmo se a distribuição das classes mudar. A quantidade total de parâmetros ajuda a contextualizar o custo de cada arquitetura.

In [ ]:
def evaluate_model(model, loader):
    model.eval()
    targets, predictions, probabilities = [], [], []

    with torch.no_grad():
        for inputs, batch_targets in loader:
            logits = model(inputs.to(device, non_blocking=True))
            batch_probabilities = torch.softmax(logits, dim=1)
            targets.extend(batch_targets.numpy())
            predictions.extend(batch_probabilities.argmax(dim=1).cpu().numpy())
            probabilities.extend(batch_probabilities.cpu().numpy())

    targets = np.asarray(targets)
    predictions = np.asarray(predictions)
    probabilities = np.asarray(probabilities)
    binary_targets = label_binarize(targets, classes=np.arange(NUM_CLASSES))

    metrics = {
        "Acurácia": accuracy_score(targets, predictions),
        "Acurácia balanceada": balanced_accuracy_score(targets, predictions),
        "Precisão macro": precision_score(targets, predictions, average="macro", zero_division=0),
        "Recall macro": recall_score(targets, predictions, average="macro", zero_division=0),
        "F1 macro": f1_score(targets, predictions, average="macro", zero_division=0),
        "ROC AUC macro": roc_auc_score(
            binary_targets, probabilities, average="macro", multi_class="ovr"
        ),
    }
    return metrics, targets, predictions, probabilities


cnn_metrics, test_targets, cnn_predictions, cnn_probabilities = evaluate_model(
    simple_cnn, test_loader
)
resnet18_metrics, _, resnet18_predictions, resnet18_probabilities = evaluate_model(
    resnet18, test_loader
)
resnet50_metrics, _, resnet50_predictions, resnet50_probabilities = evaluate_model(
    resnet50, test_loader
)

evaluated_models = [
    ("CNN simples", simple_cnn, cnn_metrics),
    ("ResNet-18", resnet18, resnet18_metrics),
    ("ResNet-50", resnet50, resnet50_metrics),
]
comparison = pd.DataFrame({
    name: {"Parâmetros (M)": parameter_counts(model)[0] / 1e6, **metrics}
    for name, model, metrics in evaluated_models
}).T
comparison.round(3)

In [ ]:
predictions_by_model = {
    "CNN simples": cnn_predictions,
    "ResNet-18": resnet18_predictions,
    "ResNet-50": resnet50_predictions,
}

for name, predictions in predictions_by_model.items():
    print(name)
    print(classification_report(
        test_targets, predictions, target_names=CLASS_NAMES, zero_division=0
    ))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 5))
for axis, (title, predictions) in zip(axes, predictions_by_model.items()):
    matrix = confusion_matrix(test_targets, predictions)
    sns.heatmap(
        matrix, annot=True, fmt="d", cmap="Blues", ax=axis,
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    )
    axis.set_title(title)
    axis.set_xlabel("Predito")
    axis.set_ylabel("Real")
    axis.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## Exemplo de inferência

As probabilidades abaixo são scores produzidos pelo modelo. Elas não foram calibradas e não representam confiança clínica.

In [ ]:
sample_index = 0
sample_record = test_frame.iloc[sample_index]
sample_tensor, sample_target = test_dataset[sample_index]

inference_models = {
    "CNN simples": simple_cnn,
    "ResNet-18": resnet18,
    "ResNet-50": resnet50,
}
sample_probabilities = {}
with torch.no_grad():
    for name, model in inference_models.items():
        model.eval()
        logits = model(sample_tensor.unsqueeze(0).to(device))
        sample_probabilities[name] = (
            torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()
        )

fig, axes = plt.subplots(1, 4, figsize=(22, 4))
image = Image.open(DATA_DIR / sample_record["relative_path"]).convert("L")
axes[0].imshow(image, cmap="gray")
axes[0].set_title(f"Classe real: {CLASS_NAMES[sample_target.item()]}")
axes[0].axis("off")
for axis, (name, probabilities) in zip(axes[1:], sample_probabilities.items()):
    sns.barplot(x=probabilities, y=CLASS_NAMES, ax=axis)
    axis.set_xlim(0, 1)
    axis.set_title(name)
    axis.set_xlabel("Probabilidade")
plt.tight_layout()
plt.show()